In [1]:
!pip install -q huggingface_hub

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# __Pipeline__

In [3]:
from huggingface_hub import login
from getpass import getpass

token = getpass("Nhập HF token: ")
login(token)

In [4]:
from transformers import pipeline

In [5]:
messages = [
    {"role": "user", "content": "Giải thích LoRA trong một câu."},
]

In [6]:
generator = pipeline(
    task="text-generation",
    model=MODEL_NAME,
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [7]:
output = generator(messages, max_new_tokens=128)
# max_new_tokens=128: giới hạn số token MỚI được sinh (khác max_length là tổng cả input).
print(output)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': [{'role': 'user', 'content': 'Giải thích LoRA trong một câu.'}, {'role': 'assistant', 'content': 'LoRA (Low-Rank Adaptation) là một kỹ thuật tối ưu hóa để fine-tuning mô hình học máy lớn một cách hiệu quả, cho phép cải thiện mô hình mà không cần lưu trữ toàn bộ trọng số được fine-tuned.'}]}]


In [8]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
14843 MiB, 15360 MiB


In [9]:
import gc, torch

del generator.model
del generator.tokenizer
del generator

gc.collect()
torch.cuda.empty_cache()

print(f"VRAM đang dùng: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"VRAM reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

VRAM đang dùng: 0.01 GB
VRAM reserved: 0.02 GB


In [11]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
155 MiB, 15360 MiB


# __Thủ công__

In [12]:
import torch
print(torch.cuda.is_bf16_supported())

True


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [15]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto"
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [16]:
messages = [{"role": "user", "content": "Viết một câu về HuggingFace."}]

In [17]:
inputs = tokenizer.apply_chat_template(
    messages, 
    add_generation_prompt=True, # dựng prompt đúng khuôn hội thoại của model (mục 5), add_generation_prompt=True để model biết “tới lượt trợ lý trả lời”.
    return_tensors="pt"
).to(model.device)

In [18]:
print(inputs)

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,  35544,  51580, 128249,
         129260, 128265,    472,  35268,  16281,     13, 151645,    198, 151644,
          77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}


In [19]:
model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [20]:
print(model.hf_device_map)

{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 0, 'model.layers.19': 0, 'model.layers.20': 0, 'model.layers.21': 0, 'model.layers.22': 0, 'model.layers.23': 0, 'model.layers.24': 0, 'model.layers.25': 'cpu', 'model.layers.26': 'cpu', 'model.layers.27': 'cpu', 'model.norm': 'cpu', 'model.rotary_emb': 'cpu', 'lm_head': 'cpu'}


In [21]:
output = model.generate(**inputs, max_new_tokens=128, use_cache=True)
new_tokens = output[0][inputs['input_ids'].shape[-1]:]
print(tokenizer.decode(new_tokens[0], skip_special_tokens=True))

H


In [26]:
print(tokenizer.decode(output, skip_special_tokens=True))

['system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.\nuser\nViết một câu về HuggingFace.\nassistant\nHuggingFace là một tổ chức mở nguồn hàng đầu cung cấp các công cụ và mô hình máy học sâu cho cộng đồng nghiên cứu và phát triển AI.']
